In [1]:
import sys

sys.path.append("..")

In [2]:
# ============================================================
# Cell 2 - Load Model & Training Dataset
# ============================================================

import torch
import torch.nn as nn

import torchvision.transforms as transforms
import torchvision.datasets as datasets

from torchvision.models import resnet18
from torch.utils.data import DataLoader

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

print("Device:", device)

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

model = resnet18(weights=None)

model.fc = nn.Linear(
    model.fc.in_features,
    10
)

checkpoint = torch.load(
    "../models/resnet18_cifar10.pth",
    map_location=device
)

model.load_state_dict(checkpoint)

model.to(device)
model.eval()

print("✓ Model Loaded Successfully")

# ------------------------------------------------------------
# CIFAR-10 Transform
# ------------------------------------------------------------

transform = transforms.Compose([

    transforms.ToTensor(),

    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2023, 0.1994, 0.2010)
    )

])

# ------------------------------------------------------------
# Training Dataset
# ------------------------------------------------------------

train_dataset = datasets.CIFAR10(

    root="../data",

    train=True,

    download=True,

    transform=transform

)

train_loader = DataLoader(

    train_dataset,

    batch_size=128,

    shuffle=False,

    num_workers=2

)

print("✓ Training Dataset Loaded")
print("Training Samples:", len(train_dataset))

Device: mps
✓ Model Loaded Successfully
✓ Training Dataset Loaded
Training Samples: 50000


In [3]:
# ============================================================
# Cell 3 - Neural State Collector
# ============================================================

from src.utils.neural_state_collector import NeuralStateCollector

target_layer = model.layer4[-1]

collector = NeuralStateCollector(target_layer)

print("✓ Neural State Collector Ready")

✓ Neural State Collector Ready


In [4]:
# ============================================================
# Cell 4 - Extract Deep Features
# ============================================================

from src.utils.compute_mahalanobis_stats import collect_features

features = collect_features(
    model=model,
    collector=collector,
    dataloader=train_loader,
    device=device
)

print("\nFeature Matrix Shape:", features.shape)

Extracting Features: 100%|██████████| 391/391 [00:15<00:00, 25.14it/s] 


Feature Extraction Complete
Samples : 50000
Feature Dimension : 512

Feature Matrix Shape: (50000, 512)


In [5]:
# ============================================================
# Cell 5 - Compute Statistics
# ============================================================

from src.utils.compute_mahalanobis_stats import (
    compute_mean,
    compute_covariance,
    compute_inverse_covariance
)

mean = compute_mean(features)

print("Mean Shape:", mean.shape)

covariance = compute_covariance(features)

print("Covariance Shape:", covariance.shape)

inverse_covariance = compute_inverse_covariance(
    covariance
)

print("Inverse Covariance Shape:",
      inverse_covariance.shape)

ImportError: cannot import name 'compute_mean' from 'src.utils.compute_mahalanobis_stats' (/Users/omvdangi/Desktop/summer internship 2026 (VU)/MetaFailurePredictor/notebooks/../src/utils/compute_mahalanobis_stats.py)

In [6]:
import importlib
import src.utils.compute_mahalanobis_stats

importlib.reload(src.utils.compute_mahalanobis_stats)

print(dir(src.utils.compute_mahalanobis_stats))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'collect_features', 'compute_covariance', 'compute_inverse_covariance', 'compute_mean', 'np', 'torch', 'tqdm']


In [7]:
import src.utils.compute_mahalanobis_stats as maha

mean = maha.compute_mean(features)
print("Mean Shape:", mean.shape)

covariance = maha.compute_covariance(features)
print("Covariance Shape:", covariance.shape)

inverse_covariance = maha.compute_inverse_covariance(covariance)
print("Inverse Covariance Shape:", inverse_covariance.shape)

Mean Shape: (512,)
Covariance Shape: (512, 512)
Inverse Covariance Shape: (512, 512)


In [8]:
import numpy as np
import os

os.makedirs("../models", exist_ok=True)

np.savez(
    "../models/mahalanobis_stats.npz",
    mean=mean,
    inv_cov=inverse_covariance
)

print("Statistics saved successfully!")

Statistics saved successfully!


In [9]:
# ============================================================
# Cell 7 - Verify Saved Statistics
# ============================================================

import numpy as np

stats = np.load("../models/mahalanobis_stats.npz")

print("Available Keys:", stats.files)

print()

print("Mean Shape:", stats["mean"].shape)
print("Inverse Covariance Shape:", stats["inv_cov"].shape)

Available Keys: ['mean', 'inv_cov']

Mean Shape: (512,)
Inverse Covariance Shape: (512, 512)
